In [1]:
import os
base_path =os.path.abspath("")
base_path = base_path + "\\"
header_path = base_path + 'headers\\header_data_treatment.ipynb'
header_path

'C:\\Users\\raphael.wargon\\Documents\\GitHub\\sorting_univ\\script\\headers\\header_data_treatment.ipynb'

In [2]:
%run $header_path

initalizing spark nlp session w gpu


In [3]:
df_works =(spark.read.format('parquet').load("file:\\" + main_path_openalex + 'works.parquet')
            .select(func.col('id').alias('work_id') ,'publication_year',func.col('cited_by_count').alias('citations'), "type",
                #func.from_json(func.col('primary_location.source'), schema_sources).alias('source'),
                    func.col('primary_location.source').alias('source'),
                'authorships','title','language','primary_location.version','primary_topic','updated_date'))

In [4]:
df_works.show()

+--------------------+----------------+---------+-------+--------------------+--------------------+--------------------+--------+----------------+--------------------+--------------------+
|             work_id|publication_year|citations|   type|              source|         authorships|               title|language|         version|       primary_topic|        updated_date|
+--------------------+----------------+---------+-------+--------------------+--------------------+--------------------+--------+----------------+--------------------+--------------------+
|https://openalex....|            2007|      345|article|{https://openalex...|[{NULL, {https://...|Barriers to the i...|      en|publishedVersion|{https://openalex...|2025-11-06T03:46:...|
|https://openalex....|            NULL|        0|  other|{https://openalex...|                  []|                    |    NULL|submittedVersion|{https://openalex...|2025-11-06T04:12:...|
|https://openalex....|            1983|        1|articl

In [5]:

works_au_af = (df_works

               .withColumn('authorships', func.explode_outer(func.col('authorships')))
            #.withColumn('authorships', func.from_json(func.col('authorships'), schema_authorships))
            .select('work_id','publication_year', 'citations', "type",func.col('source.id').alias('source_id'), func.col('source.type').alias('source_type'),
                func.col('authorships.author.id').alias('author_id'), func.col('authorships.author_position').alias('author_position'),
                func.col('authorships.raw_affiliation_strings'),
                func.col('authorships.institutions').alias('institution'), 'title','language','version',
               'primary_topic'
            )
            .withColumn('institution', func.explode_outer(func.col('institution')))
            .select(*[func.regexp_replace(func.col(id_col), 'https://openalex.org/','').alias(id_col) for id_col 
                in ['work_id','author_id','institution.id',"source_id"] ]
                ,'publication_year', 'citations', "type", 
                'author_position', #func.col('institution.id').alias('inst_id'), 
                func.col('institution.country_code').alias('country'),func.col('institution.type').alias('inst_type'),
                   'title','language','version','primary_topic', "raw_affiliation_strings")
            .withColumnRenamed('institution.id','inst_id')
               .distinct()
            )
works_au_af.printSchema()


root
 |-- work_id: string (nullable = true)
 |-- author_id: string (nullable = true)
 |-- inst_id: string (nullable = true)
 |-- source_id: string (nullable = true)
 |-- publication_year: integer (nullable = true)
 |-- citations: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- author_position: string (nullable = true)
 |-- country: string (nullable = true)
 |-- inst_type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- language: string (nullable = true)
 |-- version: string (nullable = true)
 |-- primary_topic: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- display_name: string (nullable = true)
 |    |-- subfield: struct (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- display_name: string (nullable = true)
 |    |-- field: struct (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- display_name: string (nullable = true)
 |    |-- domain: struct (nullable = true)
 |    

In [6]:
works_funding = spark.read.parquet("file:\\" + works_au_af_path.replace('works_au_af', 'works_funder'))

In [7]:
works_funding.printSchema()

root
 |-- work_id: string (nullable = true)
 |-- funder_id: string (nullable = true)
 |-- award_id: string (nullable = true)
 |-- award_funder_id: string (nullable = true)
 |-- award_external_id: string (nullable = true)
 |-- funder_name: string (nullable = true)



In [8]:
works_funding =(works_funding
                .groupBy('work_id')
                .agg(func.collect_set(func.struct(func.col('funder_id'), func.col('funder_name'))).alias('funders'),
                     func.collect_set(func.struct(func.col('award_id'), func.col('award_funder_id'), func.col('award_external_id'))).alias('awards'))
               )

In [9]:
works_funding.show()

+-----------+--------------------+--------------------+
|    work_id|             funders|              awards|
+-----------+--------------------+--------------------+
|W1000108549|[{F4320324099, Ru...|[{NULL, NULL, NULL}]|
| W100014179|[{F4320321198, Ch...|[{NULL, NULL, NULL}]|
|W1000529012|[{F4320334593, Na...|[{NULL, NULL, NULL}]|
|W1000930439|[{F4320334678, Eu...|[{G3937154688, F4...|
|W1001439608|[{F4320321001, Na...|[{G3914854039, F4...|
|W1001519207|[{F4320307778, No...|[{NULL, NULL, NULL}]|
|W1001600233|[{F4320321665, Aa...|[{NULL, NULL, NULL}]|
|W1001884611|[{F4320332183, U....|[{NULL, NULL, NULL}]|
|W1001886255|[{F4320323899, Ac...|[{NULL, NULL, NULL}]|
|W1001914737|[{F4320306076, Na...|[{NULL, NULL, NULL}]|
|W1003035256|[{F4320322581, Ve...|[{NULL, NULL, NULL}]|
|W1003042489|[{F4320324060, Ko...|[{NULL, NULL, NULL}]|
| W100376271|[{F4320315885, Au...|[{NULL, NULL, NULL}]|
|W1003778881|[{F4320322795, Mi...|[{G6773132687, F4...|
| W100508676|[{F4320320887, Eu...|[{NULL, NULL, 

In [10]:
works_au_af = (works_au_af
               .join(works_funding, on ='work_id', how = 'left')
              )

In [12]:
#works_au_af.show()

In [13]:
works_au_af.write.mode('overwrite').parquet("file:\\" + works_au_af_path)

In [16]:
works_au_af.show()

+-----------+-----------+-----------+-----------+----------------+---------+------------+---------------+-------+---------+-------------------------------------+--------+----------------+--------------------+-----------------------+-------+------+
|    work_id|  author_id|    inst_id|  source_id|publication_year|citations|        type|author_position|country|inst_type|                                title|language|         version|       primary_topic|raw_affiliation_strings|funders|awards|
+-----------+-----------+-----------+-----------+----------------+---------+------------+---------------+-------+---------+-------------------------------------+--------+----------------+--------------------+-----------------------+-------+------+
|W1000000251|A5111984641|I1292966370|       NULL|            1990|        0|book-chapter|           NULL|     GB|    other|                 Antitumor Effects...|      en|publishedVersion|{https://openalex...|   [Drug Targeting L...|   NULL|  NULL|
|W100000

In [44]:
works_au_af.write.mode('overwrite').parquet("file:\\" + works_au_af_path.replace('E:', 'D:'))

In [9]:
works_au_af_path.replace('E:', 'D:')

'D:\\openalex-snapshot\\data_extracted\\works_au_af.parquet'

In [18]:
spark.catalog.clearCache()
spark.stop()

In [14]:
works_au_af = spark.read.format('parquet').load('file:\\' +  works_au_af_path)

In [15]:
works_au_af.select('work_id', 'author_id','inst_id','author_position').groupBy('work_id','author_id','inst_id','author_position').count().groupBy(func.col('count').alias('duplicates')).count().sort('duplicates').show()

+----------+----------+
|duplicates|     count|
+----------+----------+
|         1|1089542582|
|         2|   2556726|
|         3|    503448|
|         4|    149331|
|         5|     53051|
|         6|     26003|
|         7|      9954|
|         8|      4952|
|         9|      3021|
|        10|      1876|
|        11|      1259|
|        12|       967|
|        13|       619|
|        14|       592|
|        15|      2275|
|        16|       328|
|        17|       201|
|        18|      1866|
|        19|       173|
|        20|       125|
+----------+----------+
only showing top 20 rows



In [6]:
works_au_af.select('work_id', 'author_id','inst_id').groupBy('work_id','author_id','inst_id').count().filter('count>1').show()

+-----------+-----------+-----------+-----+
|    work_id|  author_id|    inst_id|count|
+-----------+-----------+-----------+-----+
|W2953879173|A5071855816|       NULL|    2|
| W631196958|A5083161422|       NULL|    2|
|W2521131043|A5071574862|       NULL|    2|
|W3115023375|A5073479639|       NULL|    2|
|W2323576353|A5035104386|       NULL|    2|
|W2808594346|A5034623205|       NULL|    2|
|W4235592417|A5019118741|       NULL|    2|
|W4386080270|A5091485154|       NULL|    2|
|W2625102044|A5066661998|       NULL|    2|
|W1979277658|A5034644142|       NULL|    2|
|W3120761573|A5031178460|       NULL|    2|
|W4394832048|A5095697653|       NULL|    2|
|W2330181608|A5045931301|       NULL|    2|
|W4232661835|A5049054372|       NULL|    2|
|W4310427030|A5062105931|       NULL|    2|
|W3001195026|A5011839868|I4210092543|    2|
|W2909544103|A5068051556|       NULL|    2|
|W2133850356|A5070491215|       NULL|    2|
|W4317037267|A5062858694|       NULL|    2|
|W3090795230|A5032564309|I421013

In [7]:
works_au_af.filter((func.col('work_id') == 'W2953879173' )
                   &(func.col('author_id') == 'A5071855816' )
                   &(func.col('inst_id').isNull() ) ).show()

+-----------+-----------+-------+---------+----------------+---------+-------+---------------+-------+---------+--------------------+--------+-------+--------------------+
|    work_id|  author_id|inst_id|source_id|publication_year|citations|   type|author_position|country|inst_type|               title|language|version|       primary_topic|
+-----------+-----------+-------+---------+----------------+---------+-------+---------------+-------+---------+--------------------+--------+-------+--------------------+
|W2953879173|A5071855816|   NULL|     NULL|            2007|        0|article|          first|   NULL|     NULL|Use of external f...|      en|   NULL|{https://openalex...|
|W2953879173|A5071855816|   NULL|     NULL|            2007|        0|article|         middle|   NULL|     NULL|Use of external f...|      en|   NULL|{https://openalex...|
+-----------+-----------+-------+---------+----------------+---------+-------+---------------+-------+---------+--------------------+-------

In [6]:
works_au_af.show()

+-----------+---------+-------+---------+----------------+---------+------------+---------------+-------+---------+--------------------+--------+-------+-------------+
|    work_id|author_id|inst_id|source_id|publication_year|citations|        type|author_position|country|inst_type|               title|language|version|primary_topic|
+-----------+---------+-------+---------+----------------+---------+------------+---------------+-------+---------+--------------------+--------+-------+-------------+
|W2482118891|     NULL|   NULL|     NULL|            1999|        0|book-chapter|           NULL|   NULL|     NULL|      5 Case studies|    NULL|   NULL|         NULL|
|W2763351975|     NULL|   NULL|     NULL|            2013|        0|        book|           NULL|   NULL|     NULL|      Melvin Burgess|    NULL|   NULL|         NULL|
|W3005481521|     NULL|   NULL|     NULL|            2019|        2|        book|           NULL|   NULL|     NULL|             Geneses|    NULL|   NULL|       

In [13]:
works_au_af.filter(func.col('inst_id').isNotNull()).show()

+-----------+-----------+-----------+-----------+----------------+---------+-------+---------------+-------+---------+--------------------+--------+----------------+--------------------+
|    work_id|  author_id|    inst_id|  source_id|publication_year|citations|   type|author_position|country|inst_type|               title|language|         version|       primary_topic|
+-----------+-----------+-----------+-----------+----------------+---------+-------+---------------+-------+---------+--------------------+--------+----------------+--------------------+
|W4319836577|A5044825538|I1281160679| S164447516|            2023|        0|article|           last|   NULL|education|Examining the Mod...|      en|publishedVersion|                NULL|
|W3168988421|A5015903404|I1281160679|S4210172589|            2020|        0|article|         middle|   NULL|education|Role of Online – ...|      en|submittedVersion|                NULL|
|W3185121043|A5058123173|I1281160679|S4210172589|            2019

In [7]:
works_au_af.filter(func.col('author_id').isNotNull()).show()

+-----------+-----------+-------+---------+----------------+---------+------------+---------------+-------+---------+------------------------------------+--------+-------+-------------+
|    work_id|  author_id|inst_id|source_id|publication_year|citations|        type|author_position|country|inst_type|                               title|language|version|primary_topic|
+-----------+-----------+-------+---------+----------------+---------+------------+---------------+-------+---------+------------------------------------+--------+-------+-------------+
|W1000501140|A5033591997|   NULL|     NULL|            2007|        0|dissertation|          first|   NULL|     NULL|                     Balrosarío s.a.|    NULL|   NULL|         NULL|
|W1002753943|A5073374820|   NULL|     NULL|            2002|        0|     article|          first|   NULL|     NULL|从建立健全机制入手 防范和化解金融...|    NULL|   NULL|         NULL|
| W100340655|A5072844571|   NULL|     NULL|            2010|        0|     article|   

In [12]:
df_works.filter(func.col('work_id').rlike('W1005227379')).show()

+--------------------+----------------+---------+------------+--------------------+--------------------+--------------------+--------+-------+--------------------+--------------------+
|             work_id|publication_year|citations|        type|              source|         authorships|               title|language|version|       primary_topic|        updated_date|
+--------------------+----------------+---------+------------+--------------------+--------------------+--------------------+--------+-------+--------------------+--------------------+
|https://openalex....|            1990|        1|book-chapter|                NULL|[{first, {https:/...|Radikale Prostate...|    NULL|   NULL|                NULL|2023-09-24T08:11:...|
|https://openalex....|            1990|        1|book-chapter|{https://openalex...|[{first, {https:/...|Radikale Prostate...|      de|   NULL|{https://openalex...|2024-08-14T06:27:...|
+--------------------+----------------+---------+------------+-------------

In [11]:
works_au_af.select('work_id').distinct().count()

259381882

In [9]:
works_au_af.filter(func.col('primary_topic').isNull()).select('work_id').distinct().count()

247869881

In [10]:
works_au_af.filter(func.col('primary_topic').isNotNull()).select('work_id').distinct().count()

214291001

In [ ]:
works_au_af.filter(func.col('work_id').rlike('W1005227379')).collect()

In [12]:
works_au_af.select('author_id').distinct().count()

100565608

In [8]:
works_au_af.groupBy(*[func.col(col).isNull().cast('int').alias('null_' + col) for col in ['author_id','inst_id','primary_topic']] ).agg(
    func.count('work_id').alias("count"), func.mean("citations").alias('avg_cit')
).sort('count').show()


+--------------+------------+------------------+---------+------------------+
|null_author_id|null_inst_id|null_primary_topic|    count|           avg_cit|
+--------------+------------+------------------+---------+------------------+
|             1|           0|                 1|        2|               0.0|
|             1|           0|                 0|     1263|128.57561361836898|
|             1|           1|                 0| 11424472| 1.759797651917743|
|             1|           1|                 1| 24827046|0.4636311947865243|
|             0|           1|                 0|270116797|10.407572310284724|
|             0|           1|                 1|340976079| 7.389419959867625|
|             0|           0|                 1|375223890|24.436041247267067|
|             0|           0|                 0|423357791| 27.79943445519348|
+--------------+------------+------------------+---------+------------------+



In [10]:
works_au_af.count()

1445927340